In [3]:
# =============================================================
# Baseline Random Forest Model
# Features: Latitude, Longitude + date components parsed from
# Sample Date (Year, Month, Day, DayOfYear)
# Files needed:
#   - water_quality_training_dataset.csv
#   - submission_template.csv
# =============================================================

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

# -------------------------------------------------------------
# 1. LOAD DATA
# -------------------------------------------------------------
train_df = pd.read_csv("water_quality_training_dataset.csv")
test_df  = pd.read_csv("submission_template.csv")

print("Training data shape:", train_df.shape)
print(train_df.head(3))

# -------------------------------------------------------------
# 2. FEATURE ENGINEERING — parse Sample Date into components
# -------------------------------------------------------------
def parse_date_features(df):
    dates = pd.to_datetime(df['Sample Date'], format='mixed', dayfirst=True)
    df = df.copy()
    df['Year']      = dates.dt.year
    df['Month']     = dates.dt.month
    df['Day']       = dates.dt.day
    df['DayOfYear'] = dates.dt.dayofyear
    return df

train_df = parse_date_features(train_df)
test_df  = parse_date_features(test_df)

# -------------------------------------------------------------
# 3. DEFINE FEATURES & TARGETS
# -------------------------------------------------------------
FEATURE_COLS = ['Latitude', 'Longitude', 'Year', 'Month', 'Day', 'DayOfYear']
TARGET_COLS  = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

X = train_df[FEATURE_COLS]

y_TA  = train_df['Total Alkalinity']
y_EC  = train_df['Electrical Conductance']
y_DRP = train_df['Dissolved Reactive Phosphorus']

# Handle any missing values
X = X.fillna(X.median(numeric_only=True))

print(f"\nPredictor features: {FEATURE_COLS}")
print(f"Training samples:   {len(X)}\n")

# -------------------------------------------------------------
# 4. HELPER FUNCTIONS (mirrors benchmark notebook structure)
# -------------------------------------------------------------
def split_data(X, y, test_size=0.3, random_state=42):
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

def scale_data(X_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler

def train_model(X_train_scaled, y_train):
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train_scaled, y_train)
    return model

def evaluate_model(model, X_scaled, y_true, dataset_name="Test"):
    y_pred = model.predict(X_scaled)
    r2   = r2_score(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    print(f"  {dataset_name:6s} → R²: {r2:.4f} | MSE: {mse:.4f} | RMSE: {rmse:.4f}")
    return y_pred, r2, mse, rmse

# -------------------------------------------------------------
# 5. PIPELINE
# -------------------------------------------------------------
def run_pipeline(X, y, param_name="Parameter"):
    print(f"\n{'='*60}")
    print(f"  Training Model for: {param_name}")
    print(f"{'='*60}")

    X_train, X_test, y_train, y_test = split_data(X, y)
    X_train_scaled, X_test_scaled, scaler = scale_data(X_train, X_test)
    model = train_model(X_train_scaled, y_train)

    _, r2_train, mse_train, rmse_train = evaluate_model(model, X_train_scaled, y_train, "Train")
    _, r2_test,  mse_test,  rmse_test  = evaluate_model(model, X_test_scaled,  y_test,  "Test")

    results = {
        "Parameter"  : param_name,
        "R2_Train"   : round(r2_train,   4),
        "MSE_Train"  : round(mse_train,  4),
        "RMSE_Train" : round(rmse_train, 4),
        "R2_Test"    : round(r2_test,    4),
        "MSE_Test"   : round(mse_test,   4),
        "RMSE_Test"  : round(rmse_test,  4),
    }
    return model, scaler, pd.DataFrame([results])

# -------------------------------------------------------------
# 6. TRAIN ALL THREE MODELS
# -------------------------------------------------------------
model_TA,  scaler_TA,  results_TA  = run_pipeline(X, y_TA,  "Total Alkalinity")
model_EC,  scaler_EC,  results_EC  = run_pipeline(X, y_EC,  "Electrical Conductance")
model_DRP, scaler_DRP, results_DRP = run_pipeline(X, y_DRP, "Dissolved Reactive Phosphorus")

# -------------------------------------------------------------
# 7. SUMMARY TABLE
# -------------------------------------------------------------
summary = pd.concat([results_TA, results_EC, results_DRP], ignore_index=True)
summary = summary.set_index("Parameter")

print("\n")
print("=" * 70)
print("  BASELINE RANDOM FOREST — PERFORMANCE SUMMARY")
print("=" * 70)
print(summary.to_string())
print("=" * 70)

# -------------------------------------------------------------
# 8. GENERATE SUBMISSION FILE
# -------------------------------------------------------------
X_val = test_df[FEATURE_COLS].fillna(test_df[FEATURE_COLS].median(numeric_only=True))

pred_TA  = model_TA.predict(scaler_TA.transform(X_val))
pred_EC  = model_EC.predict(scaler_EC.transform(X_val))
pred_DRP = model_DRP.predict(scaler_DRP.transform(X_val))

submission_df = pd.DataFrame({
    'Longitude'                    : test_df['Longitude'].values,
    'Latitude'                     : test_df['Latitude'].values,
    'Sample Date'                  : test_df['Sample Date'].values,
    'Total Alkalinity'             : pred_TA,
    'Electrical Conductance'       : pred_EC,
    'Dissolved Reactive Phosphorus': pred_DRP,
})

submission_df.to_csv("submission_baseline_rf.csv", index=False)
print("\nSubmission file saved → submission_baseline_rf.csv")
print(submission_df.head())

Training data shape: (9319, 6)
    Latitude  Longitude Sample Date  Total Alkalinity  Electrical Conductance  \
0 -28.760833  17.730278  02-01-2011           128.912                   555.0   
1 -26.861111  28.884722  03-01-2011            74.720                   162.9   
2 -26.450000  28.085833  03-01-2011            89.254                   573.0   

   Dissolved Reactive Phosphorus  
0                           10.0  
1                          163.0  
2                           80.0  

Predictor features: ['Latitude', 'Longitude', 'Year', 'Month', 'Day', 'DayOfYear']
Training samples:   9319


  Training Model for: Total Alkalinity
  Train  → R²: 0.9798 | MSE: 111.5018 | RMSE: 10.5594
  Test   → R²: 0.8415 | MSE: 902.9410 | RMSE: 30.0490

  Training Model for: Electrical Conductance
  Train  → R²: 0.9816 | MSE: 2158.1071 | RMSE: 46.4554
  Test   → R²: 0.8623 | MSE: 16080.0510 | RMSE: 126.8071

  Training Model for: Dissolved Reactive Phosphorus
  Train  → R²: 0.9578 | MSE: 109.04

In [5]:
# =============================================================
# Enhanced Random Forest Model
# Features: Lat/Lon, date components, distance to station,
#           flow averages/lags (raw + distance-weighted),
#           and cyclical month encoding (sin/cos)
# Files needed:
#   - water_training_added_features.csv
#   - submission_template.csv
# =============================================================

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

# -------------------------------------------------------------
# 1. LOAD DATA
# -------------------------------------------------------------
train_df = pd.read_csv("water_training_added_features.csv")
test_df  = pd.read_csv("submission_template.csv")

print("Training data shape:", train_df.shape)
print(train_df.head(3))

# -------------------------------------------------------------
# 2. FEATURE ENGINEERING — parse date components
#    month_sin/cos already provided, so we only add Year and
#    DayOfYear for additional temporal signal
# -------------------------------------------------------------
def parse_date_features(df):
    dates = pd.to_datetime(df['Sample Date'], format='mixed', dayfirst=True)
    df = df.copy()
    df['Year']      = dates.dt.year
    df['DayOfYear'] = dates.dt.dayofyear
    return df

train_df = parse_date_features(train_df)
test_df  = parse_date_features(test_df)

# -------------------------------------------------------------
# 3. DEFINE FEATURES & TARGETS
# -------------------------------------------------------------
FEATURE_COLS = [
    'Latitude', 'Longitude',
    'Year', 'DayOfYear',
    'distance_to_station_km',
    'flow_7d_avg', 'flow_30d_avg',
    'flow_7d_lag', 'flow_30d_lag',
    'flow_7d_avg_weighted', 'flow_30d_avg_weighted',
    'flow_7d_lag_weighted', 'flow_30d_lag_weighted',
    'month_sin', 'month_cos',
]
TARGET_COLS = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

X = train_df[FEATURE_COLS].fillna(train_df[FEATURE_COLS].median(numeric_only=True))

y_TA  = train_df['Total Alkalinity']
y_EC  = train_df['Electrical Conductance']
y_DRP = train_df['Dissolved Reactive Phosphorus']

print(f"\nPredictor features ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"Training samples: {len(X)}\n")

# -------------------------------------------------------------
# 4. HELPER FUNCTIONS (same benchmark structure)
# -------------------------------------------------------------
def split_data(X, y, test_size=0.3, random_state=42):
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

def scale_data(X_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler

def train_model(X_train_scaled, y_train):
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train_scaled, y_train)
    return model

def evaluate_model(model, X_scaled, y_true, dataset_name="Test"):
    y_pred = model.predict(X_scaled)
    r2   = r2_score(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    print(f"  {dataset_name:6s} → R²: {r2:.4f} | MSE: {mse:.4f} | RMSE: {rmse:.4f}")
    return y_pred, r2, mse, rmse

# -------------------------------------------------------------
# 5. PIPELINE
# -------------------------------------------------------------
def run_pipeline(X, y, param_name="Parameter"):
    print(f"\n{'='*60}")
    print(f"  Training Model for: {param_name}")
    print(f"{'='*60}")

    X_train, X_test, y_train, y_test = split_data(X, y)
    X_train_scaled, X_test_scaled, scaler = scale_data(X_train, X_test)
    model = train_model(X_train_scaled, y_train)

    _, r2_train, mse_train, rmse_train = evaluate_model(model, X_train_scaled, y_train, "Train")
    _, r2_test,  mse_test,  rmse_test  = evaluate_model(model, X_test_scaled,  y_test,  "Test")

    results = {
        "Parameter"  : param_name,
        "R2_Train"   : round(r2_train,   4),
        "MSE_Train"  : round(mse_train,  4),
        "RMSE_Train" : round(rmse_train, 4),
        "R2_Test"    : round(r2_test,    4),
        "MSE_Test"   : round(mse_test,   4),
        "RMSE_Test"  : round(rmse_test,  4),
    }
    return model, scaler, pd.DataFrame([results])

# -------------------------------------------------------------
# 6. TRAIN ALL THREE MODELS
# -------------------------------------------------------------
model_TA,  scaler_TA,  results_TA  = run_pipeline(X, y_TA,  "Total Alkalinity")
model_EC,  scaler_EC,  results_EC  = run_pipeline(X, y_EC,  "Electrical Conductance")
model_DRP, scaler_DRP, results_DRP = run_pipeline(X, y_DRP, "Dissolved Reactive Phosphorus")

# -------------------------------------------------------------
# 7. SUMMARY TABLE
# -------------------------------------------------------------
summary = pd.concat([results_TA, results_EC, results_DRP], ignore_index=True)
summary = summary.set_index("Parameter")

print("\n")
print("=" * 70)
print("  ENHANCED RANDOM FOREST — PERFORMANCE SUMMARY")
print("=" * 70)
print(summary.to_string())
print("=" * 70)

# -------------------------------------------------------------
# 8. FEATURE IMPORTANCE (useful signal for next iterations)
# -------------------------------------------------------------
print("\n--- Feature Importances ---")
for name, model in [("Total Alkalinity", model_TA),
                    ("Electrical Conductance", model_EC),
                    ("Dissolved Reactive Phosphorus", model_DRP)]:
    imp = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
    print(f"\n{name}:")
    print(imp.round(4).to_string())

# -------------------------------------------------------------
# 9. GENERATE SUBMISSION FILE
#    Note: test_df won't have flow/distance features, so we
#    fill those with training medians as a neutral fallback
# -------------------------------------------------------------
flow_cols = [c for c in FEATURE_COLS if c not in ['Latitude', 'Longitude', 'Year', 'DayOfYear', 'month_sin', 'month_cos']]
train_medians = train_df[flow_cols].median()

X_val = test_df[['Latitude', 'Longitude']].copy()
X_val['Year']      = pd.to_datetime(test_df['Sample Date'], format='mixed', dayfirst=True).dt.year
X_val['DayOfYear'] = pd.to_datetime(test_df['Sample Date'], format='mixed', dayfirst=True).dt.dayofyear

# Cyclical month encoding for test set
months = pd.to_datetime(test_df['Sample Date'], format='mixed', dayfirst=True).dt.month
X_val['month_sin'] = np.sin(2 * np.pi * months / 12)
X_val['month_cos'] = np.cos(2 * np.pi * months / 12)

# Fill flow/distance features with training medians
for col in flow_cols:
    X_val[col] = train_medians[col]

X_val = X_val[FEATURE_COLS]

pred_TA  = model_TA.predict(scaler_TA.transform(X_val))
pred_EC  = model_EC.predict(scaler_EC.transform(X_val))
pred_DRP = model_DRP.predict(scaler_DRP.transform(X_val))

submission_df = pd.DataFrame({
    'Longitude'                    : test_df['Longitude'].values,
    'Latitude'                     : test_df['Latitude'].values,
    'Sample Date'                  : test_df['Sample Date'].values,
    'Total Alkalinity'             : pred_TA,
    'Electrical Conductance'       : pred_EC,
    'Dissolved Reactive Phosphorus': pred_DRP,
})

submission_df.to_csv("submission_enhanced_rf.csv", index=False)
print("\nSubmission file saved → submission_enhanced_rf.csv")
print(submission_df.head())

Training data shape: (3002, 17)
    Latitude  Longitude Sample Date  distance_to_station_km  flow_7d_avg  \
0 -33.501667  21.624167  2011-01-04               78.608612          NaN   
1 -32.502778  19.535833  2011-01-04               97.831807     0.075000   
2 -32.595833  19.009444  2011-01-06               89.112543     0.073714   

   flow_30d_avg  flow_7d_lag  flow_30d_lag  flow_7d_avg_weighted  \
0           NaN          NaN           NaN                   NaN   
1      0.073500        0.072         0.072              0.000759   
2      0.072667        0.063         0.072              0.000818   

   flow_30d_avg_weighted  flow_7d_lag_weighted  flow_30d_lag_weighted  \
0                    NaN                   NaN                    NaN   
1               0.000744              0.000729               0.000729   
2               0.000806              0.000699               0.000799   

   month_sin  month_cos  Total Alkalinity  Electrical Conductance  \
0        0.5   0.866025     

In [7]:
# =============================================================
# Enhanced Random Forest Model
# - Full 9,319-row training dataset preserved
# - Flow/distance features left-joined from enriched CSV
# - Median imputation for unmatched rows
# - max_depth + min_samples_leaf to reduce overfitting
# Files needed:
#   - water_quality_training_dataset.csv
#   - water_training_added_features.csv
#   - submission_template.csv
# =============================================================

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

# -------------------------------------------------------------
# 1. LOAD DATA
# -------------------------------------------------------------
base_df     = pd.read_csv("water_quality_training_dataset.csv")
enriched_df = pd.read_csv("water_training_added_features.csv")
test_df     = pd.read_csv("submission_template.csv")

print(f"Base training shape:     {base_df.shape}")
print(f"Enriched features shape: {enriched_df.shape}")

# -------------------------------------------------------------
# 2. NORMALISE DATES before merging
# -------------------------------------------------------------
def normalise_date(df):
    df = df.copy()
    df['Sample Date'] = pd.to_datetime(
        df['Sample Date'], format='mixed', dayfirst=True
    ).dt.strftime('%Y-%m-%d')
    return df

base_df     = normalise_date(base_df)
enriched_df = normalise_date(enriched_df)
test_df     = normalise_date(test_df)

# -------------------------------------------------------------
# 3. LEFT JOIN — keeps all 9,319 rows; NaNs where no match
# -------------------------------------------------------------
FLOW_COLS = [
    'distance_to_station_km',
    'flow_7d_avg', 'flow_30d_avg',
    'flow_7d_lag', 'flow_30d_lag',
    'flow_7d_avg_weighted', 'flow_30d_avg_weighted',
    'flow_7d_lag_weighted', 'flow_30d_lag_weighted',
    'month_sin', 'month_cos',
]

JOIN_KEYS = ['Latitude', 'Longitude', 'Sample Date']

# Only bring over the flow columns + keys from enriched_df
enriched_slim = enriched_df[JOIN_KEYS + FLOW_COLS].drop_duplicates(subset=JOIN_KEYS)

train_df = base_df.merge(enriched_slim, on=JOIN_KEYS, how='left')

matched = train_df[FLOW_COLS[0]].notna().sum()
print(f"\nRows after join:  {len(train_df)}")
print(f"Rows with flow data matched: {matched} / {len(train_df)}")

# -------------------------------------------------------------
# 4. FEATURE ENGINEERING — date components
# -------------------------------------------------------------
def parse_date_features(df):
    dates = pd.to_datetime(df['Sample Date'], format='mixed', dayfirst=True)
    df = df.copy()
    df['Year']      = dates.dt.year
    df['Month']     = dates.dt.month
    df['DayOfYear'] = dates.dt.dayofyear
    # Compute month_sin/cos — fill missing if column exists, else create fresh
    sin_vals = np.sin(2 * np.pi * dates.dt.month / 12)
    cos_vals = np.cos(2 * np.pi * dates.dt.month / 12)
    df['month_sin'] = df['month_sin'].fillna(sin_vals) if 'month_sin' in df.columns else sin_vals
    df['month_cos'] = df['month_cos'].fillna(cos_vals) if 'month_cos' in df.columns else cos_vals
    return df

train_df = parse_date_features(train_df)
test_df  = parse_date_features(test_df)


# -------------------------------------------------------------
# 5. DEFINE FEATURES & TARGETS
# -------------------------------------------------------------
FEATURE_COLS = [
    'Latitude', 'Longitude',
    'Year', 'DayOfYear',
    'distance_to_station_km',
    'flow_7d_avg', 'flow_30d_avg',
    'flow_7d_lag', 'flow_30d_lag',
    'flow_7d_avg_weighted', 'flow_30d_avg_weighted',
    'flow_7d_lag_weighted', 'flow_30d_lag_weighted',
    'month_sin', 'month_cos',
]
TARGET_COLS = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

# Impute remaining NaNs with column medians
X = train_df[FEATURE_COLS].copy()
train_medians = X.median()
X = X.fillna(train_medians)

y_TA  = train_df['Total Alkalinity']
y_EC  = train_df['Electrical Conductance']
y_DRP = train_df['Dissolved Reactive Phosphorus']

print(f"\nFinal feature count: {len(FEATURE_COLS)}")
print(f"Final training rows: {len(X)}\n")

# -------------------------------------------------------------
# 6. HELPER FUNCTIONS
# -------------------------------------------------------------
def split_data(X, y, test_size=0.3, random_state=42):
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

def scale_data(X_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler

def train_model(X_train_scaled, y_train, max_depth=15, min_samples_leaf=5):
    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_train_scaled, y_train)
    return model

def evaluate_model(model, X_scaled, y_true, dataset_name="Test"):
    y_pred = model.predict(X_scaled)
    r2   = r2_score(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    print(f"  {dataset_name:6s} → R²: {r2:.4f} | MSE: {mse:.4f} | RMSE: {rmse:.4f}")
    return y_pred, r2, mse, rmse

# -------------------------------------------------------------
# 7. PIPELINE
# -------------------------------------------------------------
def run_pipeline(X, y, param_name="Parameter", max_depth=15, min_samples_leaf=5):
    print(f"\n{'='*60}")
    print(f"  Training Model for: {param_name}")
    print(f"  max_depth={max_depth} | min_samples_leaf={min_samples_leaf}")
    print(f"{'='*60}")

    X_train, X_test, y_train, y_test = split_data(X, y)
    X_train_scaled, X_test_scaled, scaler = scale_data(X_train, X_test)
    model = train_model(X_train_scaled, y_train, max_depth, min_samples_leaf)

    _, r2_train, mse_train, rmse_train = evaluate_model(model, X_train_scaled, y_train, "Train")
    _, r2_test,  mse_test,  rmse_test  = evaluate_model(model, X_test_scaled,  y_test,  "Test")

    results = {
        "Parameter"       : param_name,
        "R2_Train"        : round(r2_train,   4),
        "MSE_Train"       : round(mse_train,  4),
        "RMSE_Train"      : round(rmse_train, 4),
        "R2_Test"         : round(r2_test,    4),
        "MSE_Test"        : round(mse_test,   4),
        "RMSE_Test"       : round(rmse_test,  4),
        "Train_Test_Gap"  : round(r2_train - r2_test, 4),
    }
    return model, scaler, pd.DataFrame([results])

# -------------------------------------------------------------
# 8. TRAIN — tweak max_depth and min_samples_leaf here
# -------------------------------------------------------------
MAX_DEPTH        = 15
MIN_SAMPLES_LEAF = 5

model_TA,  scaler_TA,  results_TA  = run_pipeline(X, y_TA,  "Total Alkalinity",             MAX_DEPTH, MIN_SAMPLES_LEAF)
model_EC,  scaler_EC,  results_EC  = run_pipeline(X, y_EC,  "Electrical Conductance",        MAX_DEPTH, MIN_SAMPLES_LEAF)
model_DRP, scaler_DRP, results_DRP = run_pipeline(X, y_DRP, "Dissolved Reactive Phosphorus", MAX_DEPTH, MIN_SAMPLES_LEAF)

# -------------------------------------------------------------
# 9. SUMMARY TABLE
# -------------------------------------------------------------
summary = pd.concat([results_TA, results_EC, results_DRP], ignore_index=True)
summary = summary.set_index("Parameter")

print("\n")
print("=" * 75)
print("  ENHANCED RF (FULL DATA + REGULARIZED) — PERFORMANCE SUMMARY")
print("=" * 75)
print(summary.to_string())
print("=" * 75)

# -------------------------------------------------------------
# 10. FEATURE IMPORTANCES
# -------------------------------------------------------------
print("\n--- Feature Importances ---")
for name, model in [("Total Alkalinity", model_TA),
                    ("Electrical Conductance", model_EC),
                    ("Dissolved Reactive Phosphorus", model_DRP)]:
    imp = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
    print(f"\n{name}:")
    print(imp.round(4).to_string())

# -------------------------------------------------------------
# 11. SUBMISSION FILE
# -------------------------------------------------------------
X_val = test_df[['Latitude', 'Longitude', 'Year', 'DayOfYear', 'month_sin', 'month_cos']].copy()

for col in [c for c in FEATURE_COLS if c not in X_val.columns]:
    X_val[col] = train_medians[col]

X_val = X_val[FEATURE_COLS]

pred_TA  = model_TA.predict(scaler_TA.transform(X_val))
pred_EC  = model_EC.predict(scaler_EC.transform(X_val))
pred_DRP = model_DRP.predict(scaler_DRP.transform(X_val))

submission_df = pd.DataFrame({
    'Longitude'                    : test_df['Longitude'].values,
    'Latitude'                     : test_df['Latitude'].values,
    'Sample Date'                  : test_df['Sample Date'].values,
    'Total Alkalinity'             : pred_TA,
    'Electrical Conductance'       : pred_EC,
    'Dissolved Reactive Phosphorus': pred_DRP,
})

submission_df.to_csv("submission_enhanced_rf_v2.csv", index=False)
print("\nSubmission file saved → submission_enhanced_rf_v2.csv")
print(submission_df.head())

Base training shape:     (9319, 6)
Enriched features shape: (3002, 17)

Rows after join:  9319
Rows with flow data matched: 3002 / 9319

Final feature count: 15
Final training rows: 9319


  Training Model for: Total Alkalinity
  max_depth=15 | min_samples_leaf=5
  Train  → R²: 0.9102 | MSE: 496.4822 | RMSE: 22.2819
  Test   → R²: 0.8449 | MSE: 883.4794 | RMSE: 29.7234

  Training Model for: Electrical Conductance
  max_depth=15 | min_samples_leaf=5
  Train  → R²: 0.9165 | MSE: 9772.5387 | RMSE: 98.8562
  Test   → R²: 0.8593 | MSE: 16421.4634 | RMSE: 128.1463

  Training Model for: Dissolved Reactive Phosphorus
  max_depth=15 | min_samples_leaf=5
  Train  → R²: 0.8059 | MSE: 501.9876 | RMSE: 22.4051
  Test   → R²: 0.6916 | MSE: 810.7973 | RMSE: 28.4745


  ENHANCED RF (FULL DATA + REGULARIZED) — PERFORMANCE SUMMARY
                               R2_Train  MSE_Train  RMSE_Train  R2_Test    MSE_Test  RMSE_Test  Train_Test_Gap
Parameter                                                     

In [8]:
# =============================================================
# Enhanced RF — CV Method Comparison + Hyperparameter Tuning
# CV methods compared:
#   - 5-Fold
#   - 10-Fold
#   - Repeated 5-Fold (5 repeats) — practical LOOCV substitute
# Then: GridSearch over max_depth and min_samples_leaf
# Files needed:
#   - water_quality_training_dataset.csv
#   - water_training_added_features.csv
#   - submission_template.csv
# =============================================================

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, KFold, RepeatedKFold,
    cross_val_score, GridSearchCV
)
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.pipeline import Pipeline

# -------------------------------------------------------------
# 1. LOAD & MERGE DATA (same as v2)
# -------------------------------------------------------------
base_df     = pd.read_csv("water_quality_training_dataset.csv")
enriched_df = pd.read_csv("water_training_added_features.csv")
test_df     = pd.read_csv("submission_template.csv")

def normalise_date(df):
    df = df.copy()
    df['Sample Date'] = pd.to_datetime(
        df['Sample Date'], format='mixed', dayfirst=True
    ).dt.strftime('%Y-%m-%d')
    return df

base_df     = normalise_date(base_df)
enriched_df = normalise_date(enriched_df)
test_df     = normalise_date(test_df)

FLOW_COLS = [
    'distance_to_station_km',
    'flow_7d_avg', 'flow_30d_avg',
    'flow_7d_lag', 'flow_30d_lag',
    'flow_7d_avg_weighted', 'flow_30d_avg_weighted',
    'flow_7d_lag_weighted', 'flow_30d_lag_weighted',
    'month_sin', 'month_cos',
]
JOIN_KEYS = ['Latitude', 'Longitude', 'Sample Date']

enriched_slim = enriched_df[JOIN_KEYS + FLOW_COLS].drop_duplicates(subset=JOIN_KEYS)
train_df = base_df.merge(enriched_slim, on=JOIN_KEYS, how='left')

def parse_date_features(df):
    dates = pd.to_datetime(df['Sample Date'], format='mixed', dayfirst=True)
    df = df.copy()
    df['Year']      = dates.dt.year
    df['DayOfYear'] = dates.dt.dayofyear
    sin_vals = np.sin(2 * np.pi * dates.dt.month / 12)
    cos_vals = np.cos(2 * np.pi * dates.dt.month / 12)
    df['month_sin'] = df['month_sin'].fillna(sin_vals) if 'month_sin' in df.columns else sin_vals
    df['month_cos'] = df['month_cos'].fillna(cos_vals) if 'month_cos' in df.columns else cos_vals
    return df

train_df = parse_date_features(train_df)
test_df  = parse_date_features(test_df)

FEATURE_COLS = [
    'Latitude', 'Longitude', 'Year', 'DayOfYear',
    'distance_to_station_km',
    'flow_7d_avg', 'flow_30d_avg',
    'flow_7d_lag', 'flow_30d_lag',
    'flow_7d_avg_weighted', 'flow_30d_avg_weighted',
    'flow_7d_lag_weighted', 'flow_30d_lag_weighted',
    'month_sin', 'month_cos',
]

X = train_df[FEATURE_COLS].copy()
train_medians = X.median()
X = X.fillna(train_medians)

targets = {
    'Total Alkalinity'             : train_df['Total Alkalinity'],
    'Electrical Conductance'       : train_df['Electrical Conductance'],
    'Dissolved Reactive Phosphorus': train_df['Dissolved Reactive Phosphorus'],
}

print(f"Training rows: {len(X)} | Features: {len(FEATURE_COLS)}\n")

# -------------------------------------------------------------
# 2. CV METHOD COMPARISON
#    Pipeline wraps scaler + model so scaling is done per fold
#    (prevents data leakage from fitting scaler on full data)
# -------------------------------------------------------------
cv_methods = {
    '5-Fold'          : KFold(n_splits=5,  shuffle=True, random_state=42),
    '10-Fold'         : KFold(n_splits=10, shuffle=True, random_state=42),
    'Repeated-5-Fold' : RepeatedKFold(n_splits=5, n_repeats=5, random_state=42),
}

base_model = RandomForestRegressor(
    n_estimators=100, max_depth=15, min_samples_leaf=5,
    random_state=42, n_jobs=-1
)

cv_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  base_model)
])

print("=" * 65)
print("  CV METHOD COMPARISON  (scoring: R²)")
print("=" * 65)

cv_results = []
for param_name, y in targets.items():
    print(f"\n{param_name}:")
    for cv_name, cv in cv_methods.items():
        scores = cross_val_score(cv_pipeline, X, y, cv=cv, scoring='r2', n_jobs=-1)
        row = {
            'Parameter' : param_name,
            'CV Method' : cv_name,
            'Mean R²'   : round(scores.mean(), 4),
            'Std R²'    : round(scores.std(),  4),
            'Min R²'    : round(scores.min(),  4),
            'Max R²'    : round(scores.max(),  4),
        }
        cv_results.append(row)
        print(f"  {cv_name:20s} → Mean: {scores.mean():.4f} | Std: {scores.std():.4f} "
              f"| Min: {scores.min():.4f} | Max: {scores.max():.4f}")

cv_summary = pd.DataFrame(cv_results)
print("\n")
print(cv_summary.to_string(index=False))

# -------------------------------------------------------------
# 3. HYPERPARAMETER GRID SEARCH
#    Uses 5-Fold CV (best speed/stability balance).
#    Searches max_depth and min_samples_leaf combinations.
# -------------------------------------------------------------
print("\n")
print("=" * 65)
print("  GRID SEARCH — max_depth x min_samples_leaf  (5-Fold CV)")
print("=" * 65)

param_grid = {
    'model__max_depth'        : [10, 15, 20, None],
    'model__min_samples_leaf' : [1, 5, 10, 20],
}

gs_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

best_params_all = {}
gs_results = []

for param_name, y in targets.items():
    print(f"\nGrid searching: {param_name} ...")
    gs = GridSearchCV(
        gs_pipeline, param_grid,
        cv=KFold(n_splits=5, shuffle=True, random_state=42),
        scoring='r2', n_jobs=-1, verbose=0
    )
    gs.fit(X, y)

    best = gs.best_params_
    best_params_all[param_name] = best
    print(f"  Best params : max_depth={best['model__max_depth']} | "
          f"min_samples_leaf={best['model__min_samples_leaf']}")
    print(f"  Best CV R²  : {gs.best_score_:.4f}")

    gs_results.append({
        'Parameter'       : param_name,
        'Best max_depth'  : best['model__max_depth'],
        'Best min_samples': best['model__min_samples_leaf'],
        'Best CV R²'      : round(gs.best_score_, 4),
    })

gs_summary = pd.DataFrame(gs_results)
print("\n")
print("=" * 65)
print("  GRID SEARCH SUMMARY")
print("=" * 65)
print(gs_summary.to_string(index=False))

# -------------------------------------------------------------
# 4. FINAL MODELS using best params from grid search
# -------------------------------------------------------------
print("\n")
print("=" * 65)
print("  FINAL MODELS WITH BEST PARAMS — TRAIN/TEST EVALUATION")
print("=" * 65)

X_train, X_test, _, _ = train_test_split(X, targets['Total Alkalinity'],
                                          test_size=0.3, random_state=42)
scaler_final = StandardScaler()
X_train_sc = scaler_final.fit_transform(X_train)
X_test_sc  = scaler_final.transform(X_test)

final_models  = {}
final_scalers = {}
final_results = []

for param_name, y in targets.items():
    bp = best_params_all[param_name]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

    sc = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr)
    X_te_sc = sc.transform(X_te)

    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=bp['model__max_depth'],
        min_samples_leaf=bp['model__min_samples_leaf'],
        random_state=42, n_jobs=-1
    )
    model.fit(X_tr_sc, y_tr)

    r2_train = r2_score(y_tr, model.predict(X_tr_sc))
    r2_test  = r2_score(y_te, model.predict(X_te_sc))
    mse_test = mean_squared_error(y_te, model.predict(X_te_sc))

    final_models[param_name]  = model
    final_scalers[param_name] = sc

    final_results.append({
        'Parameter'      : param_name,
        'max_depth'      : bp['model__max_depth'],
        'min_samples_leaf': bp['model__min_samples_leaf'],
        'R2_Train'       : round(r2_train, 4),
        'R2_Test'        : round(r2_test,  4),
        'MSE_Test'       : round(mse_test, 4),
        'RMSE_Test'      : round(np.sqrt(mse_test), 4),
        'Gap'            : round(r2_train - r2_test, 4),
    })

    print(f"\n  {param_name}  "
          f"(max_depth={bp['model__max_depth']}, min_samples_leaf={bp['model__min_samples_leaf']})")
    print(f"    Train R²: {r2_train:.4f} | Test R²: {r2_test:.4f} | "
          f"Gap: {r2_train - r2_test:.4f} | RMSE: {np.sqrt(mse_test):.4f}")

print("\n")
print(pd.DataFrame(final_results).set_index('Parameter').to_string())

# -------------------------------------------------------------
# 5. SUBMISSION FILE
# -------------------------------------------------------------
X_val = test_df[['Latitude', 'Longitude', 'Year', 'DayOfYear', 'month_sin', 'month_cos']].copy()
for col in [c for c in FEATURE_COLS if c not in X_val.columns]:
    X_val[col] = train_medians[col]
X_val = X_val[FEATURE_COLS]

submission_df = pd.DataFrame({
    'Longitude'                    : test_df['Longitude'].values,
    'Latitude'                     : test_df['Latitude'].values,
    'Sample Date'                  : test_df['Sample Date'].values,
    'Total Alkalinity'             : final_models['Total Alkalinity'].predict(
                                        final_scalers['Total Alkalinity'].transform(X_val)),
    'Electrical Conductance'       : final_models['Electrical Conductance'].predict(
                                        final_scalers['Electrical Conductance'].transform(X_val)),
    'Dissolved Reactive Phosphorus': final_models['Dissolved Reactive Phosphorus'].predict(
                                        final_scalers['Dissolved Reactive Phosphorus'].transform(X_val)),
})

submission_df.to_csv("submission_tuned_rf.csv", index=False)
print("\nSubmission file saved → submission_tuned_rf.csv")
print(submission_df.head())

Training rows: 9319 | Features: 15

  CV METHOD COMPARISON  (scoring: R²)

Total Alkalinity:
  5-Fold               → Mean: 0.8553 | Std: 0.0079 | Min: 0.8421 | Max: 0.8663
  10-Fold              → Mean: 0.8574 | Std: 0.0152 | Min: 0.8306 | Max: 0.8759
  Repeated-5-Fold      → Mean: 0.8532 | Std: 0.0108 | Min: 0.8328 | Max: 0.8708

Electrical Conductance:
  5-Fold               → Mean: 0.8652 | Std: 0.0074 | Min: 0.8533 | Max: 0.8742
  10-Fold              → Mean: 0.8707 | Std: 0.0084 | Min: 0.8581 | Max: 0.8848
  Repeated-5-Fold      → Mean: 0.8633 | Std: 0.0098 | Min: 0.8336 | Max: 0.8835

Dissolved Reactive Phosphorus:
  5-Fold               → Mean: 0.7011 | Std: 0.0083 | Min: 0.6853 | Max: 0.7099
  10-Fold              → Mean: 0.7138 | Std: 0.0222 | Min: 0.6641 | Max: 0.7368
  Repeated-5-Fold      → Mean: 0.7032 | Std: 0.0227 | Min: 0.6498 | Max: 0.7373


                    Parameter       CV Method  Mean R²  Std R²  Min R²  Max R²
             Total Alkalinity          5-Fold   0

In [9]:
# =============================================================
# Enhanced RF — n_estimators & max_features Grid Search
# Fixed settings (from prior tuning):
#   - min_samples_leaf = 5
#   - max_depth: None (TA), 15 (EC), 15 (DRP)
#   - CV: 5-Fold
# Grid search over:
#   - n_estimators : [100, 200, 500]
#   - max_features : ['sqrt', 'log2', 0.5]
# Files needed:
#   - water_quality_training_dataset.csv
#   - water_training_added_features.csv
#   - submission_template.csv
# =============================================================

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.pipeline import Pipeline

# -------------------------------------------------------------
# 1. LOAD & MERGE DATA
# -------------------------------------------------------------
base_df     = pd.read_csv("water_quality_training_dataset.csv")
enriched_df = pd.read_csv("water_training_added_features.csv")
test_df     = pd.read_csv("submission_template.csv")

def normalise_date(df):
    df = df.copy()
    df['Sample Date'] = pd.to_datetime(
        df['Sample Date'], format='mixed', dayfirst=True
    ).dt.strftime('%Y-%m-%d')
    return df

base_df     = normalise_date(base_df)
enriched_df = normalise_date(enriched_df)
test_df     = normalise_date(test_df)

FLOW_COLS = [
    'distance_to_station_km',
    'flow_7d_avg', 'flow_30d_avg',
    'flow_7d_lag', 'flow_30d_lag',
    'flow_7d_avg_weighted', 'flow_30d_avg_weighted',
    'flow_7d_lag_weighted', 'flow_30d_lag_weighted',
    'month_sin', 'month_cos',
]
JOIN_KEYS = ['Latitude', 'Longitude', 'Sample Date']

enriched_slim = enriched_df[JOIN_KEYS + FLOW_COLS].drop_duplicates(subset=JOIN_KEYS)
train_df = base_df.merge(enriched_slim, on=JOIN_KEYS, how='left')

def parse_date_features(df):
    dates = pd.to_datetime(df['Sample Date'], format='mixed', dayfirst=True)
    df = df.copy()
    df['Year']      = dates.dt.year
    df['DayOfYear'] = dates.dt.dayofyear
    sin_vals = np.sin(2 * np.pi * dates.dt.month / 12)
    cos_vals = np.cos(2 * np.pi * dates.dt.month / 12)
    df['month_sin'] = df['month_sin'].fillna(sin_vals) if 'month_sin' in df.columns else sin_vals
    df['month_cos'] = df['month_cos'].fillna(cos_vals) if 'month_cos' in df.columns else cos_vals
    return df

train_df = parse_date_features(train_df)
test_df  = parse_date_features(test_df)

FEATURE_COLS = [
    'Latitude', 'Longitude', 'Year', 'DayOfYear',
    'distance_to_station_km',
    'flow_7d_avg', 'flow_30d_avg',
    'flow_7d_lag', 'flow_30d_lag',
    'flow_7d_avg_weighted', 'flow_30d_avg_weighted',
    'flow_7d_lag_weighted', 'flow_30d_lag_weighted',
    'month_sin', 'month_cos',
]

X = train_df[FEATURE_COLS].copy()
train_medians = X.median()
X = X.fillna(train_medians)

# Fixed max_depth per parameter based on prior grid search
targets = {
    'Total Alkalinity'             : (train_df['Total Alkalinity'],             None),
    'Electrical Conductance'       : (train_df['Electrical Conductance'],        15),
    'Dissolved Reactive Phosphorus': (train_df['Dissolved Reactive Phosphorus'], 15),
}

print(f"Training rows: {len(X)} | Features: {len(FEATURE_COLS)}\n")

# -------------------------------------------------------------
# 2. GRID SEARCH — n_estimators & max_features
#    min_samples_leaf and max_depth fixed per parameter
# -------------------------------------------------------------
param_grid = {
    'model__n_estimators': [100, 200, 500],
    'model__max_features': ['sqrt', 'log2', 0.5],
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

print("=" * 65)
print("  GRID SEARCH — n_estimators x max_features  (5-Fold CV)")
print("  Fixed: min_samples_leaf=5 | max_depth per parameter")
print("=" * 65)

best_params_all = {}
gs_results      = []

for param_name, (y, max_depth) in targets.items():
    print(f"\nGrid searching: {param_name}  (max_depth={max_depth}) ...")

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model',  RandomForestRegressor(
            max_depth=max_depth,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        ))
    ])

    gs = GridSearchCV(pipeline, param_grid, cv=cv, scoring='r2', n_jobs=-1, verbose=0)
    gs.fit(X, y)

    best = gs.best_params_
    best_params_all[param_name] = best
    print(f"  Best n_estimators : {best['model__n_estimators']}")
    print(f"  Best max_features : {best['model__max_features']}")
    print(f"  Best CV R²        : {gs.best_score_:.4f}")

    # Show full 3x3 grid for transparency
    results_df = pd.DataFrame(gs.cv_results_)
    pivot = results_df.pivot_table(
        index='param_model__n_estimators',
        columns='param_model__max_features',
        values='mean_test_score'
    ).round(4)
    print(f"\n  CV R² grid (rows=n_estimators, cols=max_features):")
    print(pivot.to_string())

    gs_results.append({
        'Parameter'        : param_name,
        'max_depth'        : max_depth,
        'Best n_estimators': best['model__n_estimators'],
        'Best max_features': best['model__max_features'],
        'Best CV R²'       : round(gs.best_score_, 4),
    })

print("\n")
print("=" * 65)
print("  GRID SEARCH SUMMARY")
print("=" * 65)
print(pd.DataFrame(gs_results).set_index('Parameter').to_string())

# -------------------------------------------------------------
# 3. FINAL MODELS using best params
# -------------------------------------------------------------
print("\n")
print("=" * 65)
print("  FINAL MODELS — TRAIN/TEST EVALUATION")
print("=" * 65)

final_models  = {}
final_scalers = {}
final_results = []

for param_name, (y, max_depth) in targets.items():
    bp = best_params_all[param_name]

    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
    sc = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr)
    X_te_sc = sc.transform(X_te)

    model = RandomForestRegressor(
        n_estimators=bp['model__n_estimators'],
        max_features=bp['model__max_features'],
        max_depth=max_depth,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_tr_sc, y_tr)

    r2_train = r2_score(y_tr, model.predict(X_tr_sc))
    r2_test  = r2_score(y_te, model.predict(X_te_sc))
    mse_test = mean_squared_error(y_te, model.predict(X_te_sc))

    final_models[param_name]  = model
    final_scalers[param_name] = sc

    final_results.append({
        'Parameter'        : param_name,
        'n_estimators'     : bp['model__n_estimators'],
        'max_features'     : bp['model__max_features'],
        'max_depth'        : max_depth,
        'min_samples_leaf' : 5,
        'R2_Train'         : round(r2_train, 4),
        'R2_Test'          : round(r2_test,  4),
        'MSE_Test'         : round(mse_test, 4),
        'RMSE_Test'        : round(np.sqrt(mse_test), 4),
        'Gap'              : round(r2_train - r2_test, 4),
    })

    print(f"\n  {param_name}")
    print(f"    n_estimators={bp['model__n_estimators']} | max_features={bp['model__max_features']} | "
          f"max_depth={max_depth} | min_samples_leaf=5")
    print(f"    Train R²: {r2_train:.4f} | Test R²: {r2_test:.4f} | "
          f"Gap: {r2_train - r2_test:.4f} | RMSE: {np.sqrt(mse_test):.4f}")

print("\n")
print(pd.DataFrame(final_results).set_index('Parameter').to_string())

# -------------------------------------------------------------
# 4. FEATURE IMPORTANCES
# -------------------------------------------------------------
print("\n--- Feature Importances ---")
for param_name, model in final_models.items():
    imp = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
    print(f"\n{param_name}:")
    print(imp.round(4).to_string())

# -------------------------------------------------------------
# 5. SUBMISSION FILE
# -------------------------------------------------------------
X_val = test_df[['Latitude', 'Longitude', 'Year', 'DayOfYear', 'month_sin', 'month_cos']].copy()
for col in [c for c in FEATURE_COLS if c not in X_val.columns]:
    X_val[col] = train_medians[col]
X_val = X_val[FEATURE_COLS]

submission_df = pd.DataFrame({
    'Longitude'                    : test_df['Longitude'].values,
    'Latitude'                     : test_df['Latitude'].values,
    'Sample Date'                  : test_df['Sample Date'].values,
    'Total Alkalinity'             : final_models['Total Alkalinity'].predict(
                                         final_scalers['Total Alkalinity'].transform(X_val)),
    'Electrical Conductance'       : final_models['Electrical Conductance'].predict(
                                         final_scalers['Electrical Conductance'].transform(X_val)),
    'Dissolved Reactive Phosphorus': final_models['Dissolved Reactive Phosphorus'].predict(
                                         final_scalers['Dissolved Reactive Phosphorus'].transform(X_val)),
})

submission_df.to_csv("submission_tuned_rf_v2.csv", index=False)
print("\nSubmission file saved → submission_tuned_rf_v2.csv")
print(submission_df.head())

Training rows: 9319 | Features: 15

  GRID SEARCH — n_estimators x max_features  (5-Fold CV)
  Fixed: min_samples_leaf=5 | max_depth per parameter

Grid searching: Total Alkalinity  (max_depth=None) ...
  Best n_estimators : 500
  Best max_features : 0.5
  Best CV R²        : 0.8464

  CV R² grid (rows=n_estimators, cols=max_features):
param_model__max_features     0.5    log2    sqrt
param_model__n_estimators                        
100                        0.8451  0.7392  0.7392
200                        0.8458  0.7469  0.7469
500                        0.8464  0.7467  0.7467

Grid searching: Electrical Conductance  (max_depth=15) ...
  Best n_estimators : 500
  Best max_features : 0.5
  Best CV R²        : 0.8536

  CV R² grid (rows=n_estimators, cols=max_features):
param_model__max_features     0.5    log2    sqrt
param_model__n_estimators                        
100                        0.8531  0.7446  0.7446
200                        0.8534  0.7497  0.7497
500              